In [2]:
import os
print(os.getcwd())


/Users/matt/Desktop/Patent Litigation Analysis/notebooks


In [ ]:
#libriess
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [ ]:
#variables
pacer_cases = pd.read_csv("../data/raw/pacer_cases.csv")
attorneys = pd.read_csv("../data/raw/attorneys.csv")
documents = pd.read_csv("../data/raw/documents.csv")
names = pd.read_csv("../data/raw/names.csv")
cases = pd.read_csv("../data/raw/cases.csv")

/var/folders/51/1bmpv68x1h3gvxqw6fktjqh80000gn/T/ipykernel_54449/224244135.py:4: DtypeWarning: Columns (0: doc_number, 1: short_description, 2: upload_date) have mixed types. Specify dtype option on import or set low_memory=False.
  documents = pd.read_csv("../data/raw/documents.csv")


In [ ]:
pacer_cases.info()


<class 'pandas.DataFrame'>
RangeIndex: 74953 entries, 0 to 74952
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   case_name    74950 non-null  str  
 1   court_code   74953 non-null  str  
 2   court_name   74953 non-null  str  
 3   date_closed  68296 non-null  str  
 4   case_number  74953 non-null  str  
 5   pacer_id     74953 non-null  int64
 6   date_filed   74953 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.0 MB


In [ ]:
#datetime conversion
pacer_cases["date_filed"] = pd.to_datetime(
    pacer_cases["date_filed"],
    errors="coerce"
)

pacer_cases["date_closed"] = pd.to_datetime(
    pacer_cases["date_closed"],
    errors="coerce"
)

pacer_cases.info()

<class 'pandas.DataFrame'>
RangeIndex: 74953 entries, 0 to 74952
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   case_name    74950 non-null  str           
 1   court_code   74953 non-null  str           
 2   court_name   74953 non-null  str           
 3   date_closed  68296 non-null  datetime64[us]
 4   case_number  74953 non-null  str           
 5   pacer_id     74953 non-null  int64         
 6   date_filed   74953 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 4.0 MB


In [ ]:
#State field

US_STATES = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
    "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
    "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming", "District of Columbia", "Puerto Rico", "Virgin Islands", "Guam"
]

def extract_state(court_name):
    if pd.isna(court_name):
        return None
    for state in US_STATES:
        if re.search(state, str(court_name), re.IGNORECASE):
            return state
    return None

pacer_cases["region"] = pacer_cases["court_name"].apply(extract_state)
pacer_cases = pacer_cases.drop(columns=["state"])

print(pacer_cases["region"].value_counts())
print("\nUnmatched rows:")
print(pacer_cases[pacer_cases["region"].isna()]["court_name"].unique())

region
Texas                   12982
California              12817
Delaware                 6600
New York                 4554
Illinois                 4041
Florida                  3320
New Jersey               3127
Michigan                 2033
Massachusetts            1845
Virginia                 1826
Pennsylvania             1806
Ohio                     1774
Minnesota                1615
Georgia                  1367
Wisconsin                1246
Washington               1230
North Carolina           1074
Colorado                 1018
Utah                     1000
Missouri                  882
Connecticut               821
Maryland                  735
Arizona                   718
District of Columbia      689
Indiana                   640
Oregon                    619
Tennessee                 565
Nevada                    476
Louisiana                 446
South Carolina            328
Oklahoma                  285
Iowa                      284
Kansas                    272
Ken

In [17]:
pacer_cases.to_csv("../data/processed/pacer_cases_cleaned.csv", index=False)

In [ ]:
attorneys.head(25)

In [21]:
def parse_position(position):
    if pd.isna(position):
        return pd.Series({
            "terminated_date": None,
            "is_lead_attorney": False,
            "is_inactive": False
        })

    position = str(position)

    term_match = re.search(r"TERMINATED:\s*(\d{2}/\d{2}/\d{4})", position)
    terminated_date = term_match.group(1) if term_match else None

    is_lead_attorney = "LEAD ATTORNEY" in position.upper()

    is_inactive = "(Inactive)" in position

    return pd.Series({
        "terminated_date": terminated_date,
        "is_lead_attorney": is_lead_attorney,
        "is_inactive": is_inactive
    })

attorneys[["terminated_date", "is_lead_attorney", "is_inactive"]] = attorneys["position"].apply(parse_position)

attorneys["terminated_date"] = pd.to_datetime(attorneys["terminated_date"], format="%m/%d/%Y", errors="coerce")


In [26]:
attorneys["contactinfo"] = attorneys["contactinfo"].replace(
    to_replace=r"(?i)see above.*",
    value=None,
    regex=True
)

attorneys["contactinfo"] = attorneys.groupby("name")["contactinfo"].transform(
    lambda x: x.ffill()
)

attorneys.head(25)

,case_row_id,case_number,party_row_count,party_type,attorney_row_count,name,contactinfo,position,terminated_date,is_lead_attorney,is_inactive
0,14,0:92-cv-00398-MJP,40,Plaintiff,1,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
1,14,0:92-cv-00398-MJP,41,Plaintiff,2,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
2,14,0:92-cv-00398-MJP,42,Plaintiff,3,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
3,14,0:92-cv-00398-MJP,43,Plaintiff,4,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
4,14,0:92-cv-00398-MJP,44,Plaintiff,5,"Joel Wyman Collins , Jr","Collins and Lacy; PO Box 12487; Columbia, SC 2...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
5,14,0:92-cv-00398-MJP,45,Defendant,6,Edwin Russell Jeter,"Jeter and Williams; PO Box 7425; Columbia, SC ...",LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
6,14,0:92-cv-00398-MJP,45,Defendant,7,John Edward Cuttino,Gallivan White and Boyd; PO Box 7368; Columbia...,TERMINATED: 03/17/1994; LEAD ATTORNEY; ATTORNE...,1994-03-17,True,False
7,14,0:92-cv-00398-MJP,45,Defendant,8,Paul L Gardner,Spensley Horn Jubas and Lubitz; 1880 Century P...,LEAD ATTORNEY; ATTORNEY TO BE NOTICED,NaT,True,False
8,14,0:92-cv-00398-MJP,45,Defendant,9,R Bentz Kirby,Glenn Walters Law Firm; PO Box 1346; Orangebur...,TERMINATED: 06/05/1992; LEAD ATTORNEY; ATTORNE...,1992-06-05,True,False
9,14,0:92-cv-00398-MJP,46,Counter Claimant,10,John Edward Cuttino,Gallivan White and Boyd; PO Box 7368; Columbia...,TERMINATED: 03/17/1994; LEAD ATTORNEY; ATTORNE...,1994-03-17,True,False


In [30]:
STATE_ABBREV = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
    # Territories & DC
    "DC": "District of Columbia", "GU": "Guam", "VI": "Virgin Islands",
    "PR": "Puerto Rico", "MP": "Northern Mariana Islands"
}

def parse_contact(contact):
    if pd.isna(contact) or "see above" in str(contact).lower():
        return pd.Series({
            "firm": None,
            "address": None,
            "region": None
        })

    parts = str(contact).split(";")
    parts = [p.strip() for p in parts]

    firm = parts[0] if len(parts) > 0 else None

    region = None
    city_state_index = None
    for i, part in enumerate(parts[1:], start=1):
        state_match = re.search(r",\s*([A-Z]{2})\s+\d{5}", part)
        if state_match:
            region = STATE_ABBREV.get(state_match.group(1), state_match.group(1))
            city_state_index = i
            break

    if city_state_index:
        address = ", ".join(parts[1:city_state_index + 1])
    else:
        address = ", ".join(parts[1:3])

    return pd.Series({
        "firm": firm,
        "address": address,
        "region": region
    })

attorneys[["firm", "address", "region"]] = attorneys["contactinfo"].apply(parse_contact)

In [40]:
attorneys["party_type"] = attorneys["party_type"].str.strip().str.title()

attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bConsol\b", "Consolidated", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bDft\b", "Defendant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bCla\b", "Claimant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bPla\b", "Plaintiff", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bPty\b", "Party", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\b3Rd\b", "Third", regex=True)

attorneys["party_type"] = attorneys["party_type"].str.replace("-", " ", regex=False)

attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bThirdparty\b", "Third Party", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bCounterclaimant\b", "Counter Claimant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bCounterdefendant\b", "Counter Defendant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bCrossclaimant\b", "Cross Claimant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bCrossdefendant\b", "Cross Defendant", regex=True)

#intervenor, movant
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bIntervenor\b.*", "Intervenor", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bMovant\b.*", "Movant", regex=True)
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\bMediator\b.*", "Mediator", regex=True)

print(attorneys["party_type"].value_counts().to_string())

party_type
Defendant                             379547
Plaintiff                             284314
Counter Claimant                      248469
Counter Defendant                     247414
Consolidated Defendant                 21676
Third Party Plaintiff                   4798
Third Party Defendant                   4397
Consolidated Plaintiff                  4171
Counter Plaintiff                       3979
Consolidated Counter Claimant           3214
Movant                                  2992
Intervenor                              2919
Mediator                                2288
Consolidated Counter Defendant          1983
Cross Claimant                          1957
Cross Defendant                         1953
Interested Party                         990
Counterclaim Plaintiff                   903
Notice Only                              679
Technical Advisor                        678
Counterclaim Defendant                   558
Special Master                           533

In [41]:
attorneys["party_type"] = attorneys["party_type"].str.replace(r"\s+", " ", regex=True).str.strip()

party_type_map = {
    "Defendant Consolidated": "Consolidated Defendant",
    "Plaintiff Consolidated": "Consolidated Plaintiff",
    "Plaintiff - Consolidated": "Consolidated Plaintiff",
    "Plaintiff - Defaulted": "Consolidated Plaintiff",
}

attorneys["party_type"] = attorneys["party_type"].replace(party_type_map)
print(attorneys["party_type"].value_counts().to_string())

party_type
Defendant                             379547
Plaintiff                             284314
Counter Claimant                      248469
Counter Defendant                     247414
Consolidated Defendant                 21676
Third Party Plaintiff                   4798
Third Party Defendant                   4397
Consolidated Plaintiff                  4171
Counter Plaintiff                       3979
Consolidated Counter Claimant           3214
Movant                                  2992
Intervenor                              2919
Mediator                                2288
Consolidated Counter Defendant          1983
Cross Claimant                          1957
Cross Defendant                         1953
Interested Party                         990
Counterclaim Plaintiff                   903
Notice Only                              679
Technical Advisor                        678
Counterclaim Defendant                   558
Special Master                           533

In [51]:
party_type_map = {
    "Consolidated Counterdft": "Consolidated Counter Defendant",
    "Consolidated Countercla": "Consolidated Counter Claimant",
    "Consolidated Crossdft": "Consolidated Cross Defendant",
    "Consolidated Crossclm": "Consolidated Cross Claimant",
    "In Re:": "In Re",
    "Unknown Party Type": "Unknown",
    "Miscellaneous Party": "Miscellaneous",
    "Other Party": "Other",
    "Counterclaim Plaintiff": "Counter Plaintiff",
    "Counterclaim Defendant": "Counter Defendant",
    "Counter Claim Defendant": "Counter Defendant",
    "Adr Provider": "ADR Provider",
    "Mdl Notice": "MDL Notice",
    "4Th Party Defendant": "Fourth Party Defendant",
    "Fourthparty Plaintiff": "Fourth Party Plaintiff",
    "Ene Evaluator": "Early Neutral Evaluator",
    "Consolidated Intervenor": "Intervenor",
    "Consolidated Movant": "Movant",
    "Early Neutral Evaluator":"Mediator",
    "Neutral": "Mediator",
    "Unknown": "Miscellaneous",

}

attorneys["party_type"] = attorneys["party_type"].replace(party_type_map)
print(attorneys["party_type"].value_counts().to_string())

party_type
Defendant                             379547
Plaintiff                             284314
Counter Claimant                      248469
Counter Defendant                     247975
Consolidated Defendant                 21676
Counter Plaintiff                       4882
Third Party Plaintiff                   4798
Third Party Defendant                   4397
Consolidated Plaintiff                  4171
Consolidated Counter Claimant           3373
Movant                                  2994
Intervenor                              2956
Mediator                                2383
Consolidated Counter Defendant          2216
Cross Claimant                          1957
Cross Defendant                         1953
Interested Party                         990
Miscellaneous                            882
Notice Only                              679
Technical Advisor                        678
Special Master                           533
Respondent                               383

In [52]:
low_freq = attorneys["party_type"].value_counts()
low_freq = low_freq[low_freq <= 25].index.tolist()

keep = ["Appellant", "Appellee", "Nominal Defendant", "Relator", "Garnishee"]
low_freq = [p for p in low_freq if p not in keep]

attorneys.loc[attorneys["party_type"].isin(low_freq), "party_type"] = "Miscellaneous"

print(attorneys["party_type"].value_counts().to_string())

party_type
Defendant                             379547
Plaintiff                             284314
Counter Claimant                      248469
Counter Defendant                     247975
Consolidated Defendant                 21676
Counter Plaintiff                       4882
Third Party Plaintiff                   4798
Third Party Defendant                   4397
Consolidated Plaintiff                  4171
Consolidated Counter Claimant           3373
Movant                                  2994
Intervenor                              2956
Mediator                                2383
Consolidated Counter Defendant          2216
Cross Claimant                          1957
Cross Defendant                         1953
Interested Party                         990
Miscellaneous                            882
Notice Only                              679
Technical Advisor                        678
Special Master                           533
Respondent                               383

In [6]:
documents.info()

<class 'pandas.DataFrame'>
RangeIndex: 5186344 entries, 0 to 5186343
Data columns (total 9 columns):
 #   Column             Dtype  
---  ------             -----  
 0   case_row_id        int64  
 1   case_number        str    
 2   doc_count          int64  
 3   attachment         float64
 4   date_filed         str    
 5   long_description   str    
 6   doc_number         object 
 7   short_description  str    
 8   upload_date        str    
dtypes: float64(1), int64(2), object(1), str(5)
memory usage: 356.1+ MB


In [ ]:
print(attorneyclean["terminated_date"].head(20))

In [18]:
attorneyclean=pd.read_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/attorneys_cleaned.csv")
attorneyclean["terminated_date"] = pd.to_datetime(attorneyclean["terminated_date"], format="%Y-%m-%d", errors="coerce")
documents["date_filed"] = pd.to_datetime(documents["date_filed"], format="%m/%d/%Y", errors="coerce")
documents["upload_date"] = pd.to_datetime(documents["upload_date"], format="%m/%d/%Y", errors="coerce")
attorneyclean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1223417 entries, 0 to 1223416
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   case_row_id         1223417 non-null  int64         
 1   case_number         1223417 non-null  str           
 2   party_row_count     1223417 non-null  int64         
 3   party_type          1223416 non-null  str           
 4   attorney_row_count  1223417 non-null  int64         
 5   name                1221630 non-null  str           
 6   contactinfo         1207481 non-null  str           
 7   position            1185288 non-null  str           
 8   terminated_date     209562 non-null   datetime64[us]
 9   is_lead_attorney    1223417 non-null  bool          
 10  is_inactive         1223417 non-null  bool          
 11  firm                1207435 non-null  str           
 12  address             1167785 non-null  str           
 13  region              114

In [21]:
documents["date_filed"] = pd.to_datetime(documents["date_filed"], format="%m/%d/%Y", errors="coerce")
documents["upload_date"] = pd.to_datetime(documents["upload_date"], format="%m/%d/%Y", errors="coerce")
documents.info()

<class 'pandas.DataFrame'>
RangeIndex: 5186344 entries, 0 to 5186343
Data columns (total 9 columns):
 #   Column             Dtype        
---  ------             -----        
 0   case_row_id        int64        
 1   case_number        str          
 2   doc_count          int64        
 3   attachment         float64      
 4   date_filed         datetime64[s]
 5   long_description   str          
 6   doc_number         object       
 7   short_description  str          
 8   upload_date        datetime64[s]
dtypes: datetime64[s](2), float64(1), int64(2), object(1), str(3)
memory usage: 356.1+ MB


In [24]:
def categorize_doc(desc):
    if pd.isna(desc):
        return "Other"
    d = str(desc).strip().lower()

    if re.match(r'exhibit|exh\b|ex\.|tab [a-z0-9]|schedule [a-z0-9]|appendix', d):
        return "Exhibit"
    if re.match(r'motion', d):
        return "Motion"
    if re.match(r'order', d):
        return "Order"
    if re.match(r'brief|memorandum|memo\b', d):
        return "Brief/Memorandum"
    if re.match(r'declaration|affidavit|certification|certificate', d):
        return "Declaration/Affidavit"
    if re.match(r'notice', d):
        return "Notice"
    if re.match(r'complaint|answer|counterclaim|crossclaim', d):
        return "Complaint/Answer"
    if re.match(r'stipulation', d):
        return "Stipulation"
    if re.match(r'transcript|steno', d):
        return "Transcript"
    if re.match(r'scheduling|case management|discovery plan', d):
        return "Scheduling/Case Management"
    if re.match(r'usca|appeal|mandate', d):
        return "Appeal"
    if re.match(r'pro hac|motion for leave to appear|application.*pro hac', d):
        return "Pro Hac Vice"
    if re.match(r'summons|return of service|service', d):
        return "Summons/Service"
    if re.match(r'deposition', d):
        return "Deposition"
    if re.match(r'response|reply|opposition', d):
        return "Response/Reply"
    if re.match(r'clerk|docket|remark|case assigned|reassign', d):
        return "Administrative/Clerical"

    return "Other"

documents["doc_category"] = documents["short_description"].apply(categorize_doc)

In [25]:
documents["doc_category"].value_counts()

doc_category
Other                         4914063
Notice                          44748
Motion                          44156
Order                           43324
Exhibit                         34088
Declaration/Affidavit           26384
Response/Reply                  19101
Complaint/Answer                15435
Brief/Memorandum                10264
Stipulation                      9161
Transcript                       7482
Summons/Service                  6958
Pro Hac Vice                     3771
Administrative/Clerical          3233
Scheduling/Case Management       2262
Appeal                            981
Deposition                        933
Name: count, dtype: int64

In [26]:
documents["date_filed"] = pd.to_datetime(documents["date_filed"], format="%m/%d/%y")
documents.info()

<class 'pandas.DataFrame'>
RangeIndex: 5186344 entries, 0 to 5186343
Data columns (total 10 columns):
 #   Column             Dtype        
---  ------             -----        
 0   case_row_id        int64        
 1   case_number        str          
 2   doc_count          int64        
 3   attachment         float64      
 4   date_filed         datetime64[s]
 5   long_description   str          
 6   doc_number         object       
 7   short_description  str          
 8   upload_date        datetime64[s]
 9   doc_category       str          
dtypes: datetime64[s](2), float64(1), int64(2), object(1), str(4)
memory usage: 395.7+ MB


In [27]:
documents.to_csv("/Users/matt/Desktop/Patent Litigation Analysis/data/processed/documents_cleaned.csv", index=False)

In [31]:
names["party_type"].value_counts()

party_type
Defendant                           216436
Plaintiff                           112391
Counter Claimant                     96246
Counter Defendant                    74340
Consol Defendant                     14291
                                     ...  
witness                                  1
Intervenor - Pending                     1
Real Party In Interest Plaintiff         1
Master Commissioner                      1
Attorney Settlement Officer              1
Name: count, Length: 160, dtype: int64

In [34]:
def normalize_party_type(df, column="party_type", min_count=25):
    df = df.copy()

    # preserve original
    df[f"{column}_original"] = df[column]

    # normalize case / spacing
    df[column] = df[column].str.strip().str.title()

    # abbreviation cleanup
    df[column] = df[column].str.replace(r"\bConsol\b", "Consolidated", regex=True)
    df[column] = df[column].str.replace(r"\bDft\b", "Defendant", regex=True)
    df[column] = df[column].str.replace(r"\bCla\b", "Claimant", regex=True)
    df[column] = df[column].str.replace(r"\bPla\b", "Plaintiff", regex=True)
    df[column] = df[column].str.replace(r"\bPty\b", "Party", regex=True)
    df[column] = df[column].str.replace(r"\b3Rd\b", "Third", regex=True)
    df[column] = df[column].str.replace("-", " ", regex=False)

    # known replacements
    party_type_map = {
        "Counterclaim Plaintiff": "Counter Plaintiff",
        "Counterclaim Defendant": "Counter Defendant",
        "Unknown": "Miscellaneous",
        "Unknown Party Type": "Miscellaneous",
        "Miscellaneous Party": "Miscellaneous",
        "Other Party": "Miscellaneous",
        "Neutral": "Mediator",
        "Early Neutral Evaluator": "Mediator",
        "Adr Provider": "Provider",
        "Mdl Notice": "MDL Notice",
        "Plaintiff Consolidated": "Consolidated Plaintiff",
        "Defendant Consolidated": "Consolidated Defendant",
    }

    df[column] = df[column].replace(party_type_map)

    # whitespace cleanup
    df[column] = df[column].str.replace(r"\s+", " ", regex=True).str.strip()

    # collapse low-frequency values
    counts = df[column].value_counts()

    keep = [
        "Appellant",
        "Appellee",
        "Nominal Defendant",
        "Relator",
        "Garnishee"
    ]

    low_freq = counts[counts <= min_count].index.tolist()
    low_freq = [x for x in low_freq if x not in keep]

    df.loc[df[column].isin(low_freq), column] = "Miscellaneous"

    return df

In [35]:
names = normalize_party_type(names)
print(names["party_type"].value_counts().to_string())

party_type
Defendant                         217661
Plaintiff                         113155
Counter Claimant                  105068
Counter Defendant                  81838
Consolidated Defendant             14526
Mediator                            3672
Third Party Defendant               3423
Consolidated Plaintiff              3278
Consolidated Counter Claimant       2617
Movant                              2236
Third Party Plaintiff               2128
Intervenor                          1423
Cross Defendant                     1315
Counter Plaintiff                   1270
Cross Claimant                      1071
Consolidated Counter Defendant      1052
Special Master                       911
Miscellaneous                        869
Technical Advisor                    807
Interested Party                     781
Respondent                           362
Notice Only                          281
Amicus                               243
Appellee                             239
Provi

In [38]:
cases.head()

,case_row_id,case_number,pacer_id,case_name,court_name,assigned_to,referred_to,case_cause,jurisdictional_basis,demand,jury_demand,lead_case,related_case,settlement,date_filed,date_closed,date_last_filed
0,54973,01-Jan-1970,223949.0,"ASTRAZENECA AB et al v. SANDOZ, INC.",UNITED STATES DISTRICT COURT DISTRICT OF NEW JERSEY,Judge Joel A. Pisano,Magistrate Judge Tonianne J. Bongiovanni,35:271 Patent Infringement,Federal Question,NaN,NaN,NaN,NaN,NaN,2009-01-14,2011-06-02,NaN
1,427,0:00-cv-00019,1338.0,Banner Engineering v. Harris Instrument,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-04,2000-03-09,2000-03-02
2,428,0:00-cv-00058,1377.0,"Advanced UroScience, et al v. Inamed Corporation, et al",UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-11,2000-11-30,2001-02-28
3,429,0:00-cv-00172-DWF-AJB,NaN,Farnam Companies Inc v. Miller Manufacturing,U.S. District of Minnesota (DMN),Judge Donovan W. Frank,Chief Mag. Judge Arthur J. Boylan,35:271 Patent Infringement,Federal Question,NaN,NaN,0:98-cv-00040-DWF-AJB,"Dist of AZ, 99-01804",NaN,NaN,NaN,NaN
4,430,0:00-cv-00284,999.0,Cargo Protectors Inc v. Amerian Lock Company,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-02-07,2000-08-24,2000-10-17


In [43]:
US_STATES = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
    "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
    "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming", "District of Columbia", "Puerto Rico", "Virgin Islands", "Guam"
]

SPECIAL_COURT_MAP = {
    "Spokane": "Washington",
    "(7)": "Illinois",
}

def extract_state(court_name):
    if pd.isna(court_name):
        return None

    court_str = str(court_name)

    # special mappings first
    for keyword, state in SPECIAL_COURT_MAP.items():
        if re.search(keyword, court_str, re.IGNORECASE):
            return state

    # normal state matching
    for state in US_STATES:
        if re.search(state, court_str, re.IGNORECASE):
            return state

    return None

cases["region"] = cases["court_name"].apply(extract_state)

print(cases["region"].value_counts())
print("\nUnmatched rows:")
print(cases[cases["region"].isna()]["court_name"].unique())

region
Texas                   12941
California              12806
Delaware                 6600
New York                 4553
Illinois                 4043
Florida                  3304
New Jersey               3125
Michigan                 2020
Washington               1859
Massachusetts            1842
Pennsylvania             1806
Virginia                 1803
Ohio                     1773
Minnesota                1595
Georgia                  1367
Wisconsin                1246
North Carolina           1069
Colorado                 1018
Utah                      999
Connecticut               811
Arizona                   716
Maryland                  711
Missouri                  703
Oregon                    619
Tennessee                 565
Indiana                   497
Nevada                    474
Kansas                    450
Louisiana                 446
South Carolina            328
Iowa                      284
Oklahoma                  283
Kentucky                  213
New

In [46]:
cases["jurisdictional_basis"] = cases["jurisdictional_basis"].replace("0", np.nan)
cases["jurisdictional_basis"].value_counts()

jurisdictional_basis
Federal Question             71673
U.S. Government Defendant      528
U.S. Government Plaintiff      196
Diversity                       29
Name: count, dtype: int64

In [61]:
def assign_case_topics(text):
    if pd.isna(text):
        return pd.NA

    text = str(text).lower()
    topics = []

    if "patent" in text:
        topics.append("Patent")
    if "copyright" in text:
        topics.append("Copyright")
    if "trademark" in text:
        topics.append("Trademark")
    if "antitrust" in text:
        topics.append("Antitrust")
    if "declaratory" in text:
        topics.append("Declaratory Judgment")
    if "federal question" in text or "fed. question" in text:
        topics.append("Federal Question")
    if "diversity" in text:
        topics.append("Diversity")

    if not topics:
        return "Other"

    return "; ".join(topics)


In [62]:
cases["case_topic"] = cases["case_cause"].apply(assign_case_topics)
print(cases["case_topic"].value_counts().to_string())

case_topic
Patent                                    63412
Other                                      2341
Declaratory Judgment                       2309
Federal Question                           1334
Trademark                                  1331
Copyright                                   402
Trademark; Federal Question                 258
Diversity                                    62
Patent; Declaratory Judgment                 58
Declaratory Judgment; Diversity              39
Antitrust                                    19
Declaratory Judgment; Federal Question        9
Patent; Diversity                             3
Copyright; Declaratory Judgment               1
Antitrust; Federal Question                   1


In [50]:
print(cases["case_topic"].value_counts().to_string())

case_topic
Patent Infringement                                                                             61314
Declaratory Judgment                                                                             1295
Fed. Question                                                                                    1084
Declaratory Judgement                                                                             822
No cause code entered                                                                             762
Civil Action to Obtain Patent                                                                     702
Trademark Infringement                                                                            576
Trademark Infringement (Lanham Act)                                                               560
pt Patent Infringement                                                                            487
Copyright Infringement                                                 

In [63]:
cases.head(25)

,case_row_id,case_number,pacer_id,case_name,court_name,assigned_to,referred_to,case_cause,jurisdictional_basis,demand,jury_demand,lead_case,related_case,settlement,date_filed,date_closed,date_last_filed,region,case_topic
0,54973,01-Jan-1970,223949.0,"ASTRAZENECA AB et al v. SANDOZ, INC.",UNITED STATES DISTRICT COURT DISTRICT OF NEW JERSEY,Judge Joel A. Pisano,Magistrate Judge Tonianne J. Bongiovanni,35:271 Patent Infringement,Federal Question,NaN,NaN,NaN,NaN,NaN,2009-01-14,2011-06-02,NaN,New Jersey,Patent
1,427,0:00-cv-00019,1338.0,Banner Engineering v. Harris Instrument,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-04,2000-03-09,2000-03-02,Minnesota,NaN
2,428,0:00-cv-00058,1377.0,"Advanced UroScience, et al v. Inamed Corporation, et al",UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-11,2000-11-30,2001-02-28,Minnesota,NaN
3,429,0:00-cv-00172-DWF-AJB,NaN,Farnam Companies Inc v. Miller Manufacturing,U.S. District of Minnesota (DMN),Judge Donovan W. Frank,Chief Mag. Judge Arthur J. Boylan,35:271 Patent Infringement,Federal Question,NaN,NaN,0:98-cv-00040-DWF-AJB,"Dist of AZ, 99-01804",NaN,NaN,NaN,NaN,Minnesota,Patent
4,430,0:00-cv-00284,999.0,Cargo Protectors Inc v. Amerian Lock Company,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-02-07,2000-08-24,2000-10-17,Minnesota,NaN
5,431,0:00-cv-00295,1010.0,"Multi-Tech Systems v. Gateway 2000, Inc, et al",UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-02-08,2003-03-27,2005-02-23,Minnesota,NaN
6,432,0:00-cv-00296,1011.0,Multi-Tech Systems v. Dell Computer Corp,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,Judge Ann D. Montgomery,Chief Mag. Judge Raymond L. Erickson,35:145 Patent Infringement,Federal Question,NaN,NaN,NaN,NaN,NaN,2000-02-08,2003-03-27,2005-01-27,Minnesota,Patent
7,433,0:00-cv-00297,1012.0,Multi-Tech Systems v. Compaq Computer,UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,Judge Ann D. Montgomery,Chief Mag. Judge Raymond L. Erickson,35:145 Patent Infringement,Federal Question,NaN,NaN,NaN,NaN,NaN,2000-02-08,2003-03-27,2005-01-18,Minnesota,Patent
8,434,0:00-cv-00315,1030.0,"Wagner Spray Tech v. EZ Painter, Inc.",UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-02-09,2002-02-19,2002-02-19,Minnesota,NaN
9,435,0:00-cv-00335,1050.0,"Minnesota Mining, et al v. Kosma-Kare Intl",UNITED STATES DISTRICT COURT DISTRICT OF MINNESOTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000-02-11,2000-06-19,2000-06-16,Minnesota,NaN


In [72]:
demand_map = {
    "P": "Plaintiff",
    "p": "Plaintiff",
    "D": "Defendant",
    "d": "Defendant",
    "B": "Both",
    "b": "Both",
    "N": np.nan,
    "n": np.nan,
    "y": np.nan,
    "Y": np.nan,
}
cases["demand"] = cases["demand"].replace(
    r"^\s*\$[\d,]+(?:\.\d{2})?\s*$",
    "Monetary",
    regex=True
)
cases["demand"] = cases["demand"].replace(demand_map)
cases["demand"].value_counts()

demand
Plaintiff    31673
Both         15978
Defendant     3848
Monetary       283
Name: count, dtype: int64

In [75]:
print(cases["settlement"].value_counts().to_string())

settlement
5:98-cv-20428-JF; 5:98-cv-20678-JF; 5:98-cv-20698-JF                                                                                                                                                                                                                                                                                                                                                                                                                                                                               7
3:96-cv-00163-WHO2; 3:96-cv-03994-WHO2                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [ ]:
cases

In [76]:
cases.to_csv("../data/processed/cases_cleaned.csv", index=False)